<a href="https://colab.research.google.com/github/GreatLakesCommission/IEDRR_inland_lakes/blob/main/get_spp_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
do not run - copy to page console to prevent accidental disconnections


function ClickConnect() {
  console.log('Working')
  document
    .querySelector('#top-toolbar > colab-connect-button')
    .shadowRoot.querySelector('#connect')
    .click()
}
intervalTiming = setInterval(ClickConnect, 60000)

'''

In [1]:
# general setup
%%capture
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("drive/My Drive/iedrr")
today = datetime.date.today().strftime('%Y%m%d')
outfolder = "speciesobs_"+today
if not os.path.exists(outfolder):
  os.mkdir(outfolder)
os.chdir(outfolder)
import pandas as pd
!pip install pyreadr
import pyreadr
from scipy import spatial


Mounted at /content/drive


NameError: name 'datetime' is not defined

In [41]:
# invasive species of interest


fishlist = {
    'sci_name': ['Channa', 'Clarias batrachus', 'Gymnocephalus cernua', 'Misgurnus anguillicaudatus', 'Neogobius melanostomus', 'Osmerus mordax', 'Osteoglossum bicirrhosum', 'Proterorhinus semilunaris', 'Tinca tinca'],
    'common_name': ['Snakeheads', 'Walking catfish', 'Ruffe', 'Pond loach', 'Round goby', 'Rainbow smelt', 'Silver arowana', 'Tubenose goby', 'Tench']
}
fishlist = pd.DataFrame(data=fishlist)
fishlist['taxon'] = 'fish'


plantlist = {
    'sci_name': ['Alternanthera philoxeroides', 'Cabomba caroliniana', 'Callitriche stagnalis', 'Elodea densa', 'Hottonia palustris', 'Hydrilla verticillata', 'Hydrocharis morsus-ranae', 'Hygrophila polysperma', 'Limnophila sessiliflora', 'Ludwigia grandiflora', 'Ludwigia hexapetala', 'Ludwigia peploides', 'Marsilea mutica', 'Marsilea quadrifolia', 'Myriophyllum aquaticum', 'Najas minor', 'Nasturtium officinale', 'Nelumbo nucifera', 'Nitellopsis obtusa', 'Ottelia alismoides', 'Pistia stratiotes', 'Pontederia azurea', 'Pontederia crassipes', 'Sagittaria sagittifolia', 'Salvinia auriculata', 'Salvinia molesta', 'Spirodela punctata', 'Stratiotes aloides', 'Trapa natans'],
    'common_name': ['Alligator weed', 'Carolina fanwort', 'Pond water-starwort', 'Brazilian waterweed', 'Water violet', 'Hydrilla', 'European frog-bit', 'Indian swampweed', 'Dwarf ambulia', 'Large-flower primrose-willow', 'Six petal water primrose', 'Creeping water primrose', 'Australian water-clover', 'European water-clover', 'Parrot feather', 'Brittle naiad', 'Water-cress', 'Sacred lotus', 'Starry stonewort', 'Duck-lettuce', 'Water lettuce', 'Anchored water-hyacinth', 'Common water-hyacinth', 'Hawaii arrowhead', 'Eared salvinia', 'Giant salvinia', 'Dotted duckweed', 'Water soldier', 'European water chestnut']
}

# plantlist_slim doesn't include emergent species
plantlist_slim = {
    'sci_name': ['Cabomba caroliniana', 'Callitriche stagnalis', 'Elodea densa', 'Hydrilla verticillata', 'Hydrocharis morsus-ranae', 'Hygrophila polysperma', 'Limnophila sessiliflora', 'Marsilea mutica', 'Marsilea quadrifolia', 'Myriophyllum aquaticum', 'Najas minor', 'Nelumbo nucifera', 'Nitellopsis obtusa', 'Ottelia alismoides', 'Pistia stratiotes', 'Pontederia azurea', 'Pontederia crassipes', 'Salvinia auriculata', 'Salvinia molesta', 'Spirodela punctata', 'Stratiotes aloides', 'Trapa natans'],
    'common_name': ['Carolina fanwort', 'Pond water-starwort', 'Brazilian waterweed', 'Hydrilla', 'European frog-bit', 'Indian swampweed', 'Dwarf ambulia', 'Australian water-clover', 'European water-clover', 'Parrot feather', 'Brittle naiad', 'Sacred lotus', 'Starry stonewort', 'Duck-lettuce', 'Water lettuce', 'Anchored water-hyacinth', 'Common water-hyacinth', 'Eared salvinia', 'Giant salvinia', 'Dotted duckweed', 'Water soldier', 'European water chestnut']
}

plantlist = pd.DataFrame(data=plantlist)
plantlist['taxon'] = 'plant'


invertlist = {
    'sci_name': ['Bithynia tentaculata', 'Bythotrephes longimanus', 'Cercopagis pengoi', 'Corbicula fluminea', 'Dreissena bugensis', 'Dreissena polymorpha', 'Eriocheir sinensis', 'Hemimysis anomala', 'Melanoides tuberculata', 'Potamopyrgus antipodarum', 'Procambarus virginalis'],
    'common_name': ['Faucet snail', 'Spiny water flea', 'Fishhook waterflea', 'Basket clam', 'Quagga mussel', 'Zebra mussel', 'Mitten crab', 'Bloody red shrimp', 'Red-rimmed melania', 'New Zealand mud snail', 'Marbled crayfish (Marmorkrebs)']
}

invertlist = pd.DataFrame(data=invertlist)
invertlist['taxon'] = 'invertebrate'


# get institution lat/longs for QAQC
url = "https://github.com/ropensci/CoordinateCleaner/raw/refs/heads/master/data/institutions.rda"
dst_path = os.path.join(os.getcwd(), "institutions.rda") # download to CoLab
res = pyreadr.read_r(pyreadr.download_file(url, dst_path)) # convert rda to dictionary of dataframes

institutions = res["institutions"]
# filter to institutions within GL bounding box
institutions = institutions[(institutions['decimalLatitude'].between(36.9171,49.6117)) & (institutions['decimalLongitude'].between(-100.5513,-71.79))]
# set up a k-dimensional tree
inst_coords = list(zip(institutions["decimalLatitude"], institutions["decimalLongitude"]))
tree = spatial.KDTree(inst_coords)

def calculate_min(row):
    return tree.query([(row["decimalLatitude"],row["decimalLongitude"])])[0][0]

!rm institutions.rda

GBIF

In [9]:
#GBIF setup
%%capture
!pip install pygbif
# capture suppresses output

#setup
from pygbif import species as species
from pygbif import occurrences as occ
from pygbif.occurrences.download import GbifDownload
import os
import glob
import datetime
from time import sleep
import zipfile
# add your gbif.org username, password and contact email for download notices to Colab's 'Secrets'
# toggle notebook access on for all three
from google.colab import userdata
%env GBIF_USER=userdata.get('GBIF_USER')
%env GBIF_PWD = userdata.get('GBIF_PWD')
%env GBIF_EMAIL = userdata.get('GBIF_EMAIL')



SLEEP_DURATION = 20


In [23]:
# download GBIF species obs since 1970 within GL bounding box as zip files via API
my_vars = {}

my_vars["fish"] = fishlist
my_vars["plant"] = plantlist
my_vars["invert"] = invertlist

gbif_obs = {}

skip = [] #skip taxa you previously exported a csv for

# get GBIF taxon IDs based on scientific names
def getskey(z):
  return species.name_backbone(z)['usageKey']

for i, (k, v) in enumerate(my_vars.items()):
  if not k in skip:
    records = []
    print("downloading ", k)
    splist = v['sci_name'].tolist()
    splist = list(set(splist))
    spkeys = [ getskey(x) for x in splist ]
    spkeys = list(map(str, spkeys))
    #res = occurrences.download(*args, **kwargs, hasCoordinate=True, hasGeospatialIssue=False, decimalLatitude='36.9171,49.6117', decimalLongitude='-100.5513,-71.79', year="1970,2025", pred_type='and', basisOfRecord = ['HUMAN_OBSERVATION', 'OBSERVATION', 'MACHINE_OBSERVATION', 'LIVING_SPECIMEN', 'MATERIAL_SAMPLE'])

    #res = paginated_search(500000, key=skey, data='children')
    # construct query
    gbif_query = GbifDownload(userdata.get('GBIF_USER'), userdata.get('GBIF_EMAIL'))
    gbif_query.add_predicate_dict({"type": "in", "key": "BASIS_OF_RECORD", "values": ['HUMAN_OBSERVATION', 'OBSERVATION', 'MACHINE_OBSERVATION', 'LIVING_SPECIMEN', 'MATERIAL_SAMPLE'], "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "equals", "key": 'HAS_COORDINATE', 'value': 'TRUE', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "equals", "key": 'HAS_GEOSPATIAL_ISSUE', 'value': 'FALSE', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "within", "geometry": "POLYGON((-100.551 36.917,-71.79 36.917,-71.79 49.612,-100.551 49.612,-100.551 36.917))"})
    gbif_query.add_predicate_dict({"type": "greaterThanOrEquals", "key": 'YEAR', 'value': '1970', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "in", "key": 'TAXON_KEY', 'values': spkeys, "matchCase": "false"})
    # submit download query
    xx = gbif_query.post_download(userdata.get('GBIF_USER'), userdata.get('GBIF_PWD'))
    # wait for download to be ready
    while True:
      print(f"waiting to get download {xx}...")
      status = occ.download_meta(key = xx)['status']

      if status not in ['PREPARING', 'RUNNING']:  # = not ready yet
          if status == 'SUCCEEDED':
              print(f"Download is ready, getting it")
              output_path = k+"_gbif_obs"
              if os.path.exists(output_path): # get rid of any previous downloads for this run
                files = glob.glob(output_path+'/*.zip')
                for f in files:
                  os.remove(f)
              else:
                os.mkdir(output_path)

              occ.download_get(xx, output_path)
          else:
              print(f"Status is {status}, why?")
              print(occ.download_meta(key = xx))
          break

      sleep(SLEEP_DURATION)

    print("finished with ", k)


downloading  fish
waiting to get download 0071788-241126133413365...
Download is ready, getting it
finished with  fish
downloading  plant
waiting to get download 0071795-241126133413365...
Download is ready, getting it
finished with  plant
downloading  invert
waiting to get download 0071800-241126133413365...
Download is ready, getting it
finished with  invert


In [34]:
    # read the GBIF data back in and QA/QC it
    for k in ['fish', 'plant', 'invert']:
      # read zip file into dataframe
      output_path = k+"_gbif_obs"
      files = os.listdir(output_path)
      file_path = os.path.join(output_path, files[0])
      print(file_path)
      base_name, extension = os.path.splitext(files[0])
      zf = zipfile.ZipFile(file_path)
      df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')

      print(len(df.index), k, " records")

      #drop observations whose coordinates are the location of an institution
      df["mindist"] = df.apply(calculate_min, axis=1)


      df = df[df["mindist"] > 0.00000000001] # buffer in degrees

      # drop duplicate locations within each species
      df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))

      # export to csv before continuing because this cell will take forever
      df.to_csv(output_path + '/'+ k +'_obs_gbifraw_' + datetime.date.today().strftime('%Y%m%d') + '.csv', index=False)
      # add to dict
      gbif_obs[k] = df



fish_gbif_obs/0071788-241126133413365.zip


<ipython-input-34-436bc3795b9a>:10: DtypeWarning: Columns (17,29,36,37,38,39,40,41,43,44,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')


49881 fish  records


<ipython-input-34-436bc3795b9a>:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))


plant_gbif_obs/0071795-241126133413365.zip


<ipython-input-34-436bc3795b9a>:10: DtypeWarning: Columns (39,46) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')


19345 plant  records


<ipython-input-34-436bc3795b9a>:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))


invert_gbif_obs/0071800-241126133413365.zip
8874 invert  records


<ipython-input-34-436bc3795b9a>:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('taxonKey').apply(lambda x: x.drop_duplicates(['decimalLatitude', 'decimalLongitude']))


GLANSIS

In [38]:
import requests

url = "https://nas.er.usgs.gov/ipt/archive.do?r=nas_glansis"
filename = "GLANSIS_{}.zip".format(today)  # Choose a name for the downloaded file

response = requests.get(url)

if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print("Zip file downloaded successfully.")
else:
    print("Failed to download the zip file.")

Zip file downloaded successfully.


In [56]:
zf = zipfile.ZipFile(filename)
df = pd.read_csv(zf.open('occurrence.txt'), sep='\t')

df['eventDate'] = pd.to_datetime(df['eventDate'], format="%Y-%m-%d", errors='coerce')
# Drop rows with invalid dates
df = df.dropna(subset='eventDate')

#QAQC
df = df.loc[(df['decimalLatitude'] >= 36.917) & (df['decimalLatitude'] <= 49.612) & (df['decimalLongitude'] >= -100.551) & (df['decimalLongitude'] <= -71.79)] # bounding box
df = df.loc[df['eventDate']>datetime.datetime(1970,1,1)] # drop old data
df = df.loc[df['georeferenceRemarks'] != "Centroid"] # drop obs with poor coordinates

my_vars = {}
glansis_obs = {}

my_vars["fish"] = fishlist
my_vars["plant"] = plantlist
my_vars["invert"] = invertlist

for i, (k, v) in enumerate(my_vars.items()):
  splist = v['sci_name'].tolist()
  splist = list(set(splist))
  if "Elodea densa" in splist:
    splist.append("Egeria densa") # GLANSIS is using Egeria densa instead of Elodea densa
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)

  # drop nontarget species
  ndf = df[(df["scientificName"].isin(splist)) | (df["genus"].isin(hightax))]

  glansis_obs[k] = ndf

In [57]:
glansis_obs['fish']

,id,modified,language,bibliographicCitation,references,collectionID,basisOfRecord,dynamicProperties,occurrenceID,catalogNumber,...,georeferenceRemarks,taxonID,scientificName,kingdom,order,family,genus,specificEpithet,scientificNameAuthorship,vernacularName
1,urn:USGS:NAS:35633,1989-12-13,en,United States Geological Survey. 2023. Nonindi...,http://nas.er.usgs.gov/queries/SpecimenViewer....,USGS-NAS 35633,Occurrence,NaN,urn:USGS:NAS:35633,35633,...,Approximate,http://www.itis.gov/servlet/SingleRpt/SingleRp...,Gymnocephalus cernua,Animalia,Perciformes,Percidae,Gymnocephalus,cernua,"(Linnaeus, 1758)",Ruffe
2,urn:USGS:NAS:164465,2005-06-13,en,United States Geological Survey. 2023. Nonindi...,http://nas.er.usgs.gov/queries/SpecimenViewer....,USGS-NAS 164465,Occurrence,NaN,urn:USGS:NAS:164465,164465,...,Approximate,http://www.itis.gov/servlet/SingleRpt/SingleRp...,Gymnocephalus cernua,Animalia,Perciformes,Percidae,Gymnocephalus,cernua,"(Linnaeus, 1758)",Ruffe
14,urn:USGS:NAS:46611,1999-06-16,en,United States Geological Survey. 2023. Nonindi...,http://nas.er.usgs.gov/queries/SpecimenViewer....,USGS-NAS 46611,Occurrence,NaN,urn:USGS:NAS:46611,46611,...,Approximate,http://www.itis.gov/servlet/SingleRpt/SingleRp...,Gymnocephalus cernua,Animalia,Perciformes,Percidae,Gymnocephalus,cernua,"(Linnaeus, 1758)",Ruffe
15,urn:USGS:NAS:46579,1999-06-17,en,United States Geological Survey. 2023. Nonindi...,http://nas.er.usgs.gov/queries/SpecimenViewer....,USGS-NAS 46579,Occurrence,NaN,urn:USGS:NAS:46579,46579,...,Approximate,http://www.itis.gov/servlet/SingleRpt/SingleRp...,Gymnocephalus cernua,Animalia,Perciformes,Percidae,Gymnocephalus,cernua,"(Linnaeus, 1758)",Ruffe
16,urn:USGS:NAS:46580,1999-06-17,en,United States Geological Survey. 2023. Nonindi...,http://nas.er.usgs.gov/queries/SpecimenViewer....,USGS-NAS 46580,Occurrence,NaN,urn:USGS:NAS:46580,46580,...,Approximate,http://www.itis.gov/servlet/SingleRpt/SingleRp...,Gymnocephalus cernua,Animalia,Perciformes,Percidae,Gymnocephalus,cernua,"(Linnaeus, 1758)",Ruffe
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92626,urn:USGS:NAS:1711810,2023-04-07,en,United States Geological Survey. 2023. Nonindi...,http://nas.er.usgs.gov/queries/SpecimenViewer....,USGS-NAS 1711810,Occurrence,USFWS AIS Early Detection and Monitoring surve...,urn:USGS:NAS:1711810,1711810,...,Accurate,http://www.itis.gov/servlet/SingleRpt/SingleRp...,Gymnocephalus cernua,Animalia,Perciformes,Percidae,Gymnocephalus,cernua,"(Linnaeus, 1758)",Ruffe
92627,urn:USGS:NAS:1711811,2023-04-07,en,United States Geological Survey. 2023. Nonindi...,http://nas.er.usgs.gov/queries/SpecimenViewer....,USGS-NAS 1711811,Occurrence,USFWS AIS Early Detection and Monitoring surve...,urn:USGS:NAS:1711811,1711811,...,Accurate,http://www.itis.gov/servlet/SingleRpt/SingleRp...,Gymnocephalus cernua,Animalia,Perciformes,Percidae,Gymnocephalus,cernua,"(Linnaeus, 1758)",Ruffe
93259,urn:USGS:NAS:1713689,2023-06-28,en,United States Geological Survey. 2023. Nonindi...,http://nas.er.usgs.gov/queries/SpecimenViewer....,USGS-NAS 1713689,Occurrence,NaN,urn:USGS:NAS:1713689,1713689,...,Accurate,http://www.itis.gov/servlet/SingleRpt/SingleRp...,Neogobius melanostomus,Animalia,Perciformes,Gobiidae,Neogobius,melanostomus,"(Pallas, 1814)",Round Goby
93260,urn:USGS:NAS:1713690,2023-06-28,en,United States Geological Survey. 2023. Nonindi...,http://nas.er.usgs.gov/queries/SpecimenViewer....,USGS-NAS 1713690,Occurrence,NaN,urn:USGS:NAS:1713690,1713690,...,Accurate,http://www.itis.gov/servlet/SingleRpt/SingleRp...,Neogobius melanostomus,Animalia,Perciformes,Gobiidae,Neogobius,melanostomus,"(Pallas, 1814)",Round Goby


In [53]:
df['eventDate'][171]

'1983-12-31'

MISIN

In [11]:
!pip install esri2gpd


In [35]:
import esri2gpd

# MISIN observations layer updated daily
url = "https://services.arcgis.com/uHAHKfH1Z5ye1Oe0/arcgis/rest/services/misin_database_obs/FeatureServer/0"

my_vars = {}
misin_obs = {}

my_vars["fish"] = fishlist
my_vars["plant"] = plantlist
my_vars["invert"] = invertlist

# convert esri date to datetime
def convert_esri_date(row):
    """Converts an esriFieldTypeDate value to a Python datetime object."""
    return datetime.datetime.fromtimestamp(row["DAY"] / 1000)  # Divide by 1000 to get seconds

for i, (k, v) in enumerate(my_vars.items()):
  # separate list of genera
  splist = v['sci_name'].tolist()
  splist = list(set(splist))
  # separate out genera with no species epithet
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)
  if "Elodea densa" in splist:
    splist.append("Egeria densa") # MISIN is using Egeria densa instead of Elodea densa
  genus = [x.split()[0] for x in splist]

  gdf = esri2gpd.get(url, fields=['RECORDID', 'OBSERVER', 'DAY', 'LATITUDE', 'LONGITUDE', 'GENUS', 'SPECIES', 'VERIFIED'], where=f"GENUS IN {tuple(genus)}")
  gdf["DAY"] = gdf.apply(convert_esri_date, axis=1)
  #QAQC
  gdf = gdf[gdf["VERIFIED"] == 2]  # for field VERIFIED, 2 = 'Trusted Source', only keep these
  gdf = gdf.loc[(gdf['LATITUDE'] >= 36.917) & (gdf['LATITUDE'] <= 49.612) & (gdf['LONGITUDE'] >= -100.551) & (gdf['LONGITUDE'] <= -71.79)] # bounding box
  gdf = gdf.loc[gdf['DAY']>datetime.datetime(1970,1,1)] # drop old data
  # drop nontarget species
  gdf["sci_name"] = gdf["GENUS"] + " " + gdf["SPECIES"]
  gdf = gdf[(gdf["sci_name"].isin(splist)) | (gdf["GENUS"].isin(hightax))]

  misin_obs[k] = gdf

In [36]:
misin_obs['fish']

,geometry,OBSERVER,DAY,LATITUDE,LONGITUDE,GENUS,SPECIES,VERIFIED,sci_name
1,POINT (-82.98477 42.55559),Jennifer Johnson,2013-06-26 05:00:00,42.555590,-82.984770,Neogobius,melanostomus,2,Neogobius melanostomus
2,POINT (-82.99066 44.02062),Jennifer Johnson,2013-10-22 05:00:00,44.020620,-82.990660,Neogobius,melanostomus,2,Neogobius melanostomus
3,POINT (-86.137 42.32671),Jennifer Johnson,2013-10-22 05:00:00,42.326710,-86.136820,Neogobius,melanostomus,2,Neogobius melanostomus
5,POINT (-83.56916 44.14357),Janice Frey,2014-08-28 05:00:00,44.143568,-83.569157,Neogobius,melanostomus,2,Neogobius melanostomus
6,POINT (-83.52646 44.25839),Janice Frey,2014-08-28 05:00:00,44.258386,-83.526456,Neogobius,melanostomus,2,Neogobius melanostomus
...,...,...,...,...,...,...,...,...,...
1426,POINT (-86.29447 43.21722),William Keiper,2020-08-27 05:00:00,43.217221,-86.294468,Neogobius,melanostomus,2,Neogobius melanostomus
1427,POINT (-85.78884 42.29838),Brian Gunderman,2022-02-25 05:00:00,42.298377,-85.788841,Neogobius,melanostomus,2,Neogobius melanostomus
1434,POINT (-85.21898 44.95161),Caroline Keson,2021-12-06 05:00:00,44.951606,-85.218981,Neogobius,melanostomus,2,Neogobius melanostomus
1732,POINT (-86.20538 42.77287),Anne Schmieder,2024-08-09 05:00:00,42.772873,-86.205379,Neogobius,melanostomus,2,Neogobius melanostomus


iMapInvasives

In [ ]:
https://imapinvasives.natureserve.org/arcgis/rest/services/public_presence/MapServer/4



EDDMapS

In [ ]:
import requests

s = requests.session()
url = 'https://www.eddmaps.org/tools/index.cfm?forcelogin&'
login_data = {'username': userdata.get('EDDMAPS_USER'),
                  'password': userdata.get('EDDMAPS_PWD')}
res1 = s.post(url, login_data)
try:

    res1.raise_for_status()
except Exception as e:
    print('login failed')

# Make a query
url2 = 'https://www.eddmaps.org/tools/query/results.cfm?reporter=&userGroupID=&observationDateStart=&observationDateEnd=&dateEnteredStart=&dateEnteredEnd=&dateUpdatedStart=&dateUpdatedEnd=&objectid=&subjectnumber=&cat=&div=&eradicationstatus=2&list=&rank=&habitat=&country=926&state=&fipscode=&township=&layersourceid=&project='
res2 = s.get(url2)
try:
    res2.raise_for_status()
except Exception as e:
    print('query failed')


Combine

In [ ]:
one dataframe - species, latlong,date,source, recordID

drop duplicates

save a csv for each taxon

In [ ]:
# get GBIF backbone taxonomic keys for species of interest
#   taxon keys are issued to accepted names with synonyms of those accepted names
#   issued the same identifier

#splist = fishlist['sci_name'].tolist() + plantlist['sci_name'].tolist() + invertlist['sci_name'].tolist()
for lst in [fishlist, plantlist, invertlist]:
  splist = lst['sci_name'].tolist()
  splist = list(set(splist))
  keys = [ species.name_backbone(x)['usageKey'] for x in splist ]
  out = [ occ.search(taxonKey = x, limit=0)['count'] for x in keys ]
  x = dict(zip(splist, out))
  print(sorted(x.items(), key=lambda z:z[1], reverse=True))

  for x in keys:
    occ.search(taxonKey = x)

  out = [ occ.search(taxonKey = x) for x in keys ]
  [ x['results'][0]['speciesKey'] for x in out ]


[('Gymnocephalus cernua', 328363), ('Tinca tinca', 187233), ('Neogobius melanostomus', 58955), ('Osmerus mordax', 51845), ('Misgurnus anguillicaudatus', 37760), ('Channidae', 21146), ('Proterorhinus semilunaris', 14780), ('Clarias batrachus', 2400), ('Osteoglossum bicirrhosum', 698)]


KeyboardInterrupt: 

In [ ]:
for count, lst in enumerate([fishlist, plantlist, invertlist]):
  splist = lst['sci_name'].tolist()
  splist = list(set(splist))
  keys = [ species.name_backbone(x)['usageKey'] for x in splist ]
  out = [ occ.search(taxonKey = x, limit=0)['count'] for x in keys ]
  x = dict(zip(splist, out))
  print(sorted(x.items(), key=lambda z:z[1], reverse=True))
  #records = occ.search(taxonKey in splist, format='SIMPLE_CSV', hasCoordinate=True, year="1970,2025", pred_type='and')

  # Convert the results to a DataFrame
  #df = pd.DataFrame(records['results'])


[('Gymnocephalus cernua', 328365), ('Tinca tinca', 187233), ('Neogobius melanostomus', 58957), ('Osmerus mordax', 51845), ('Misgurnus anguillicaudatus', 37760), ('Channidae', 21146), ('Proterorhinus semilunaris', 14780), ('Clarias batrachus', 2400), ('Osteoglossum bicirrhosum', 698)]
[('Hydrocharis morsus-ranae', 113798), ('Nasturtium officinale', 107080), ('Sagittaria sagittifolia', 90314), ('Hottonia palustris', 77291), ('Callitriche stagnalis', 63395), ('Stratiotes aloides', 44850), ('Alternanthera philoxeroides', 39635), ('Pontederia crassipes', 39192), ('Ludwigia peploides', 28908), ('Ludwigia grandiflora', 24832), ('Pistia stratiotes', 23183), ('Myriophyllum aquaticum', 21904), ('Trapa natans', 20200), ('Hydrilla verticillata', 18728), ('Ludwigia hexapetala', 16221), ('Nitellopsis obtusa', 13586), ('Najas minor', 9833), ('Marsilea quadrifolia', 9371), ('Elodea densa', 8925), ('Nelumbo nucifera', 6619), ('Salvinia molesta', 6567), ('Cabomba caroliniana', 5973), ('Spirodela punctat